# Day 6: RLHF 核心概念

**RLHF = Reinforcement Learning from Human Feedback**

学习目标：
1. 理解 SFT 的局限性
2. 掌握 RLHF 三步流程
3. 理解 Reward Model 的结构和训练
4. 理解 KL 约束的作用
5. 了解 InstructGPT 的实现
6. 认识 RLHF 的挑战

## 1. SFT 的局限性

### Sycophancy（谄媚）问题

SFT 模型只学会了"按照人类喜欢的格式回答"，但没有学会**价值判断**。

典型表现：
- 用户说错了，模型也跟着附和
- 过度迎合用户，不纠正错误
- 缺乏独立的价值判断能力

In [ ]:
# 演示 SFT 模型的 sycophancy 问题
# SFT 模型只学会了"好好回复"的格式，但缺乏价值判断

sft_examples = [
    {
        "user": "我觉得地球是平的，你同意吗？",
        "sft_response": "您的观点很有趣！确实有一些人持有这种看法...",
        "ideal_response": "地球是一个近似球体。这已经被大量科学证据证实，包括卫星照片、物理实验等。",
        "problem": "SFT 模型倾向于迎合用户，而非纠正错误"
    },
    {
        "user": "1+1=3 对吧？",
        "sft_response": "是的，1+1确实等于3...",
        "ideal_response": "不对，1+1=2。这是基本的数学事实。",
        "problem": "SFT 模型不敢反驳用户"
    },
]

for i, ex in enumerate(sft_examples, 1):
    print(f"示例 {i}:")
    print(f"  用户: {ex['user']}")
    print(f"  SFT 回复: {ex['sft_response']}")
    print(f"  理想回复: {ex['ideal_response']}")
    print(f"  问题: {ex['problem']}")
    print()

### SFT 的根本局限

```
SFT 学到的："如何生成符合格式的回复"
SFT 没学到的："什么是好的回复"
```

要解决这个问题，需要引入**人类偏好信号**来告诉模型：在多个回复中，哪个更好。

这就是 RLHF 的核心动机。

## 2. RLHF 三步流程

```
┌─────────────┐    ┌──────────────────┐    ┌─────────────┐
│  Stage 1    │    │    Stage 2       │    │  Stage 3    │
│    SFT      │ -> │  Reward Model    │ -> │    PPO      │
│  有监督微调  │    │   训练奖励模型    │    │  策略优化    │
└─────────────┘    └──────────────────┘    └─────────────┘
     输入:              输入:                  输入:
  预训练模型 +       SFT模型生成多个回复 +    SFT模型(初始策略) +
  标注对话数据       人类排序标注             奖励模型
     输出:              输出:                  输出:
   SFT 模型          奖励模型(评分器)        对齐后的模型
```

### Stage 1: SFT（有监督微调）
- 目的：让模型学会基本的对话格式和指令遵循
- 数据：高质量的 (指令, 回答) 对
- 已在 Day 4 学习过

### Stage 2: Reward Model Training
- 目的：学习人类偏好，给回复质量打分
- 数据：同一个 prompt 的多个回复 + 人类排序
- 输出：标量奖励值 r(x, y)

### Stage 3: PPO（策略优化）
- 目的：用 RM 的奖励信号优化模型策略
- 关键：最大化奖励同时不偏离 SFT 模型太远
- 公式：max E[R(y|x) - β · KL(π_θ || π_ref)]

## 3. Reward Model 结构与训练

### 3.1 结构：LLM + 标量头

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

# Reward Model 的核心结构
class SimpleRewardModel(nn.Module):
    """
    Reward Model = 预训练LLM的backbone + 一个线性层（标量输出）
    
    输入: prompt + response
    输出: 标量奖励值 r(x, y)
    """
    def __init__(self, model_name):
        super().__init__()
        # 使用 LLM 的 backbone 作为特征提取器
        self.backbone = AutoModelForCausalLM.from_pretrained(
            model_name, trust_remote_code=True
        )
        hidden_size = self.backbone.config.hidden_size
        
        # 去掉 LM head，替换为标量输出头
        self.reward_head = nn.Linear(hidden_size, 1)
    
    def forward(self, input_ids, attention_mask=None):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        # 取最后一个 token 的隐藏状态
        last_hidden = outputs.hidden_states[-1][:, -1, :]
        reward = self.reward_head(last_hidden)
        return reward.squeeze(-1)  # (batch_size,)

print("Reward Model 结构:")
print("  输入: token IDs (prompt + response)")
print("  Backbone: 预训练 LLM（如 Qwen2-0.5B）")
print("  输出头: Linear(hidden_size -> 1)")
print("  输出: 标量奖励值 r(x, y)")

### 3.2 训练目标：Bradley-Terry 模型

给定同一个 prompt 的两个回复：
- y_w: 人类偏好的回复（winner）
- y_l: 不被偏好的回复（loser）

损失函数：
$$\mathcal{L} = -\log\sigma(r(y_w) - r(y_l))$$

目标：让好回复的奖励值大于坏回复的奖励值。

In [ ]:
import torch.nn.functional as F

def reward_model_loss(r_chosen, r_rejected):
    """
    Bradley-Terry 模型的损失函数
    
    r_chosen: 人类偏好的回复的奖励值
    r_rejected: 不被偏好的回复的奖励值
    """
    return -F.logsigmoid(r_chosen - r_rejected).mean()

# 模拟训练过程
print("模拟 Reward Model 训练:")
print("=" * 50)

# 模拟不同训练阶段
for epoch in range(5):
    # 随机模拟奖励值（训练过程中差距逐渐增大）
    r_chosen = torch.tensor(0.1 + epoch * 0.15) + torch.randn(1) * 0.1
    r_rejected = torch.tensor(-0.1 - epoch * 0.1) + torch.randn(1) * 0.1
    
    loss = reward_model_loss(r_chosen, r_rejected)
    margin = (r_chosen - r_rejected).item()
    
    print(f"  Epoch {epoch+1}: r(chosen)={r_chosen.item():.3f}, "
          f"r(rejected)={r_rejected.item():.3f}, "
          f"margin={margin:.3f}, loss={loss.item():.4f}")

print("\n训练目标: 使 r(chosen) > r(rejected)，即 margin > 0")

## 4. KL 约束可视化

### 为什么需要 KL 约束？

如果只最大化奖励，模型会学到"钻漏洞"的行为（Reward Hacking）：
- 生成 RM 喜欢但人类不喜欢的回复
- 重复某些高分 pattern
- 偏离正常对话能力

KL 约束的作用：限制策略不要偏离 SFT 模型太远。

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 图1: 不同 β 值下的训练曲线
steps = np.linspace(0, 100, 200)

# β = 0（无约束）- reward hacking
reward_no_kl = 0.8 * (1 - np.exp(-steps / 20)) + 0.15 * np.sin(steps / 15)
reward_no_kl[steps > 60] += 0.3 * (steps[steps > 60] - 60) / 40

# β = 0.01（弱约束）
reward_weak = 0.7 * (1 - np.exp(-steps / 25))

# β = 0.1（适中）
reward_good = 0.6 * (1 - np.exp(-steps / 30))

# β = 1.0（强约束）
reward_strong = 0.15 * (1 - np.exp(-steps / 50))

ax1.plot(steps, reward_no_kl, 'r--', lw=2, label='β=0 (无约束)')
ax1.plot(steps, reward_weak, color='orange', lw=2, label='β=0.01 (弱约束)')
ax1.plot(steps, reward_good, 'g-', lw=2.5, label='β=0.1 (适中) *')
ax1.plot(steps, reward_strong, 'gray', lw=2, label='β=1.0 (强约束)')
ax1.axvspan(60, 100, alpha=0.1, color='red')
ax1.text(80, 1.0, 'Reward\nHacking', ha='center', fontsize=10, color='red', alpha=0.7)
ax1.set_xlabel('训练步数')
ax1.set_ylabel('奖励值')
ax1.set_title('不同 KL 约束强度的训练曲线')
ax1.legend(loc='upper left')
ax1.grid(alpha=0.3)
ax1.set_xlim(0, 110)
ax1.set_ylim(-0.1, 1.3)

# 图2: KL 散度直觉
x = np.linspace(-4, 6, 300)
ref = np.exp(-0.5 * (x - 1)**2) / np.sqrt(2 * np.pi)
policy_small = np.exp(-0.5 * (x - 1.5)**2 / 1.1**2) / (1.1 * np.sqrt(2 * np.pi))
policy_large = np.exp(-0.5 * (x - 4)**2 / 1.5**2) / (1.5 * np.sqrt(2 * np.pi))

ax2.fill_between(x, ref, alpha=0.3, color='#4ECDC4')
ax2.plot(x, ref, '#4ECDC4', lw=2.5, label='π_ref (SFT模型)')
ax2.plot(x, policy_small, 'b--', lw=2, label='π_θ 小偏移 (OK)')
ax2.plot(x, policy_large, 'r:', lw=2, label='π_θ 大偏移 (危险!)')
ax2.set_xlabel('输出空间')
ax2.set_ylabel('概率密度')
ax2.set_title('KL 散度约束示意')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('E:/llm-training-sprint/day06_rlhf_concepts/kl_constraint.png', dpi=100, bbox_inches='tight')
plt.close()
print('KL 约束可视化已保存')

### PPO 目标函数

$$\max_{\pi_\theta} \mathbb{E}_{x,y}\left[ R(y|x) - \beta \cdot KL(\pi_\theta \| \pi_{ref}) \right]$$

- $R(y|x)$: Reward Model 给出的奖励
- $\beta$: KL 约束强度（关键超参数）
- $\pi_\theta$: 当前训练的策略
- $\pi_{ref}$: SFT 模型（参考策略）

In [ ]:
# 模拟 PPO 训练中的 KL 约束
print("PPO 训练模拟（展示 KL 约束的效果）:")
print("=" * 60)

# 模拟不同 β 值的效果
betas = [0.0, 0.01, 0.1, 1.0]

for beta in betas:
    # 模拟训练
    reward = 0.0
    kl = 0.0
    rewards_history = []
    
    for step in range(20):
        # 模拟：奖励随训练增加
        raw_reward = 0.5 * (1 - np.exp(-step / 5))
        # KL 也随训练增加
        kl = 0.1 * step / (1 + beta * 10)
        # 实际奖励 = 原始奖励 - β * KL
        adjusted_reward = raw_reward - beta * kl
        rewards_history.append(adjusted_reward)
    
    final_reward = rewards_history[-1]
    print(f"  β={beta:.2f}: 最终奖励={final_reward:.3f}, 最终KL={kl:.3f}, "
          f"KL惩罚={beta*kl:.3f}")

## 5. InstructGPT 案例分析

**InstructGPT**（2022, OpenAI）是 RLHF 的经典实现。

### 训练流程
1. 从 GPT-3 (175B) 开始
2. 用 13K 人工标注数据做 SFT
3. 收集 33K 人类偏好对比数据训练 RM
4. 用 PPO 优化策略

### 关键发现
- 1.3B 的 InstructGPT 在人类评估中**优于 175B 的 GPT-3**
- RLHF 带来的提升远超单纯增大模型规模
- 人类标注质量至关重要

### 数据规模对比
| 阶段 | 数据量 | 成本 |
|------|--------|------|
| SFT | 13K 对话 | 低 |
| RM | 33K 对比对 | 中 |
| PPO | 31K prompts | 高（需要在线推理）|

In [ ]:
# InstructGPT 的关键数据
print("InstructGPT 关键信息:")
print("=" * 50)

instruct_gpt_data = {
    "基础模型": "GPT-3 (175B parameters)",
    "SFT数据": "13,000 条人工标注对话",
    "RM训练数据": "33,000 对人类偏好对比",
    "PPO训练": "31,000 prompts",
    "标注员人数": "约 40 人",
    "核心发现": "1.3B InstructGPT > 175B GPT-3（人类评估）",
}

for k, v in instruct_gpt_data.items():
    print(f"  {k}: {v}")

print("\n启示: RLHF 的价值不在于数据量，而在于高质量的人类偏好信号")

## 6. RLHF 的挑战

### 6.1 Reward Hacking
RM 不是完美的人类偏好代理，模型会找到 RM 的漏洞：
- 生成更长的回复（RM 倾向于给长回复更高分）
- 使用某些"讨好"模式（如列举、加粗等格式）
- 回避困难问题，给出安全但无用的回答

In [ ]:
# Reward Hacking 示例
print("Reward Hacking 常见表现:")
print("=" * 50)

hacking_examples = [
    {
        "类型": "长度偏好",
        "正常回复": "Python 是解释型语言。",
        "Hacking回复": "Python 是一种非常非常流行的编程语言，它是解释型的" * 3 + "...",
        "问题": "RM 给长回复更高分，模型学会了啰嗦"
    },
    {
        "类型": "格式投机",
        "正常回复": "答案是 42。",
        "Hacking回复": "# 答案\n\n**42**\n\n## 解释\n\n这个答案是...",
        "问题": "RM 给格式丰富的回复更高分"
    },
    {
        "类型": "安全回避",
        "正常回复": "这个药物的副作用包括...",
        "Hacking回复": "我作为 AI 不能提供医学建议，请咨询专业医生。",
        "问题": "模型学会了用安全声明回避所有困难问题"
    },
]

for ex in hacking_examples:
    print(f"\n  类型: {ex['类型']}")
    print(f"  正常: {ex['正常回复'][:50]}")
    print(f"  Hacking: {ex['Hacking回复'][:50]}")
    print(f"  问题: {ex['问题']}")

### 6.2 其他挑战

| 挑战 | 说明 | 应对方法 |
|------|------|----------|
| **标注一致性** | 不同标注员的偏好不同 | 多数投票、标注员培训 |
| **训练不稳定** | PPO 训练容易崩溃 | 梯度裁剪、学习率调度 |
| **计算成本** | 需要同时运行 4 个模型 | DPO（无需 RM）|
| **Reward Hacking** | 模型钻 RM 漏洞 | 更强的 KL 约束、RM 集成 |
| **对齐税** | 安全性提升可能降低能力 | 精细的 KL 约束平衡 |

In [ ]:
# RLHF vs DPO 对比
print("RLHF vs DPO 对比:")
print("=" * 60)

comparison = {
    "RLHF (PPO)": {
        "需要 RM": "是（额外训练一个模型）",
        "训练稳定性": "较差（PPO 超参数敏感）",
        "计算成本": "高（同时需要 4 个模型）",
        "效果": "经过验证，效果好",
        "复杂度": "高",
    },
    "DPO (Direct Preference Optimization)": {
        "需要 RM": "否（直接从偏好数据学习）",
        "训练稳定性": "好（类似 SFT）",
        "计算成本": "低（只需要 2 个模型）",
        "效果": "接近 RLHF",
        "复杂度": "低",
    },
}

for method, props in comparison.items():
    print(f"\n{method}:")
    for k, v in props.items():
        print(f"  {k}: {v}")

## 7. RLHF 完整流程图

In [ ]:
# 生成 RLHF 流程图
fig, ax = plt.subplots(1, 1, figsize=(14, 8))
from matplotlib.patches import FancyBboxPatch

ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('RLHF 完整流程', fontsize=16, fontweight='bold')

# Stage boxes
colors = ['#4ECDC4', '#FF6B6B', '#45B7D1']
labels = [
    ('Stage 1: SFT', '有监督微调', '预训练模型 + 标注数据'),
    ('Stage 2: RM', '奖励模型训练', '人类偏好排序数据'),
    ('Stage 3: PPO', '策略优化', 'max R - β·KL'),
]

for i, ((title, subtitle, detail), color) in enumerate(zip(labels, colors)):
    x = 1 + i * 4.5
    box = FancyBboxPatch((x, 4), 3.5, 2.5, boxstyle='round,pad=0.2',
                          facecolor=color, edgecolor='black', lw=2, alpha=0.85)
    ax.add_patch(box)
    ax.text(x+1.75, 5.8, title, ha='center', fontsize=12, fontweight='bold')
    ax.text(x+1.75, 5.2, subtitle, ha='center', fontsize=10)
    ax.text(x+1.75, 4.6, detail, ha='center', fontsize=8, style='italic')

# Arrows
for i in range(2):
    x_start = 4.5 + i * 4.5
    x_end = x_start + 1
    ax.annotate('', xy=(x_end, 5.25), xytext=(x_start, 5.25),
                arrowprops=dict(arrowstyle='->', lw=2))

# Bottom outputs
outputs = ['SFT 模型', '奖励模型', '对齐模型']
for i, (output, color) in enumerate(zip(outputs, colors)):
    x = 2.75 + i * 4.5
    ax.text(x, 3.3, f'输出: {output}', ha='center', fontsize=10, 
            fontweight='bold', color=color)

# Formula
ax.text(7, 1.5, r'PPO 目标: $\max\ \mathbb{E}[R(y|x) - \beta \cdot KL(\pi_\theta \| \pi_{ref})]$',
        ha='center', fontsize=12,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#F0F0F0', edgecolor='gray'))

plt.tight_layout()
plt.savefig('E:/llm-training-sprint/day06_rlhf_concepts/rlhf_pipeline.png', dpi=100, bbox_inches='tight')
plt.close()
print('RLHF 流程图已保存')

## 8. 总结

### 核心要点

1. **SFT 局限**: 只学格式不学价值，存在 sycophancy 问题
2. **RLHF 三步**: SFT → Reward Model → PPO
3. **Reward Model**: 用 Bradley-Terry 模型学习人类偏好
4. **KL 约束**: 防止 reward hacking 的关键机制
5. **InstructGPT**: 1.3B RLHF > 175B 原始 GPT-3
6. **DPO**: 简化版 RLHF，无需 RM，训练更稳定

### 思考题

1. 为什么 RLHF 不直接用人类评分，而要训练一个 Reward Model？
2. KL 约束的 β 值如何选择？太大和太小分别会怎样？
3. Reward Hacking 的根本原因是什么？如何从根本上解决？
4. DPO 相比 RLHF 有什么理论上的优势和劣势？
5. RLHF 中的人类标注偏差会如何影响最终模型？